In [0]:
ENV = dbutils.widgets.get("env") if dbutils.widgets.get("env") else "dev"

In [0]:
df = spark.read.table("dev.bronze.yellow_taxi_raw")
display(df)

In [0]:
# counting null values
from pyspark.sql.functions import col, sum, when

null_counts = df.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns])
null_counts.show()


# checking for duplicates
total_rows = df.count()
unique_rows = df.dropDuplicates(["VendorID", "tpep_pickup_datetime", "PULocationID"]).count()
duplicate_count = total_rows - unique_rows
print (f"Duplicate rows: {duplicate_count}")


# checking for invalid values
negative_fares = df.filter(col("fare_amount") <= 0).count()
invalid_distance = df.filter(col("trip_distance") <= 0).count()

print(f"Rows with negative fares: {negative_fares}")
print(f"Rows with invalid distance: {invalid_distance}")

In [0]:
# deduplicate
df = df.dropDuplicates(["VendorID", "tpep_pickup_datetime", "PULocationID"])

In [0]:
# creating a new column 'is_valid' 
df = df.withColumn("is_valid", 
    when(
        (col("passenger_count").isNull()) | 
        (col("passenger_count") == 0) |
        (col("trip_distance") <= 0) | 
        (col("fare_amount") < 0) |
        (col("tpep_pickup_datetime") < "2023-01-01") |
        (col("tpep_pickup_datetime") >= "2023-02-01") |
        (col("PULocationID").isNull()) |
        (col("DOLocationID").isNull()) |
        (col("Ratecodeid").isNull()),
        False
    ).otherwise(True))

In [0]:
# creating a new column is_valid_time
df = df.withColumn("is_valid_time",
    when(
        col("tpep_dropoff_datetime") < col("tpep_pickup_datetime"),
        False
    ).otherwise(True))

In [0]:
df = df.fillna({'congestion_surcharge': 0, 'airport_fee': 0})

In [0]:
# renaming columns
new_columns = ["vendor_id", "pickup_datetime", "dropoff_datetime", "passenger_count","trip_distance", "ratecode_id", "store_and_fwd_flag", "pu_location_id", "do_location_id", "payment_type", "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount", "improvement_surcharge", "total_amount", "congestion_surcharge", "airport_fee", "ingestion_timestamp", "source_file", "is_valid", "is_valid_time"]
df = df.toDF(*new_columns)
display(df)

In [0]:
# creating silver table
from delta.tables import DeltaTable

if not spark.catalog.tableExists("dev.silver.yellow_taxi_clean"):
    df.write.format("delta").saveAsTable("dev.silver.yellow_taxi_clean")
else:
    DeltaTable.forName(spark, "dev.silver.yellow_taxi_clean") \
    .alias("existing") \
    .merge(
        df.alias("new"),
        "existing.vendor_id = new.vendor_id AND existing.pickup_datetime = new.pickup_datetime AND existing.pu_location_id = new.pu_location_id"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

In [0]:
spark.sql("SELECT COUNT(*) FROM dev.silver.yellow_taxi_clean").show()